# Model Performance Comparison — NB02 + NB03

Aggregates CNN encoder and authenticator metrics across all completed dataset runs.
Datasets whose result files are missing are skipped automatically.

In [1]:
import re, json, sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

sys.path.insert(0, '../..')

DATASETS     = ['whuGAIT', 'ucihar', 'wisdm', 'combined']
TEX_DIR      = Path('../../latex/generated')
EXECUTED_DIR = Path('../../executed')
OUT_DIR      = Path('../../results/analysis')
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEX_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {'whuGAIT': '#3498db', 'ucihar': '#e74c3c', 'wisdm': '#2ecc71', 'combined': '#9b59b6'}

def _nb_text(nb_path):
    if not Path(nb_path).exists():
        return ''
    nb = json.load(open(nb_path))
    parts = []
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        for out in cell.get('outputs', []):
            parts.append(''.join(out.get('text', '')))
    return '\n'.join(parts)

def _float(text, pattern, default=np.nan):
    m = re.search(pattern, text)
    return float(m.group(1)) if m else default

def _int(text, pattern, default=0):
    m = re.search(pattern, text)
    return int(m.group(1)) if m else default

def load_ds_metrics(ds, executed_dir):
    nb02 = _nb_text(executed_dir / ds / '02_train_cnn_encoder.ipynb')
    nb03 = _nb_text(executed_dir / ds / '03_train_authenticator.ipynb')
    if not nb02 and not nb03:
        return None
    return {
        'cnn_subjects':    _int(nb02,   r'Training subjects:\s+(\d+)'),
        'cnn_acc':         _float(nb02, r'Best test accuracy:\s+([\d.]+)%'),
        'cnn_best_epoch':  _int(nb02,   r'Best epoch:\s+(\d+)/'),
        'auth_best_epoch': _int(nb03,   r'Best epoch:\s+(\d+)/'),
        'auth_test_acc':   _float(nb03, r'Test accuracy:\s+([\d.]+)%'),
        'auth_test_auc':   _float(nb03, r'\nAUC:\s+([\d.]+)'),
        'auth_same_mean':  _float(nb03, r'Same person:\s+mean P\(different\) = ([\d.]+)'),
        'auth_diff_mean':  _float(nb03, r'Different person:\s+mean P\(different\) = ([\d.]+)'),
        'auth_sep':        _float(nb03, r'Separation:\s+([\d.]+)'),
    }

def parse_epoch_curves(nb_path, pattern):
    if not Path(nb_path).exists():
        return []
    nb = json.load(open(nb_path))
    rows = []
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        for out in cell.get('outputs', []):
            txt = ''.join(out.get('text', ''))
            for line in txt.split('\n'):
                m = re.search(pattern, line)
                if m:
                    rows.append({k: float(v) for k, v in m.groupdict().items()})
    return rows

## Architecture diagrams

Generate CNN encoder and AuthModel architecture figures saved to `latex/thesis/figures/`.

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

FIG_DIR = Path('../../latex/thesis/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

C_INPUT  = '#2c3e50'
C_CONV   = '#2980b9'
C_POOL   = '#16a085'
C_LSTM   = '#d35400'
C_FC     = '#8e44ad'
C_OUT    = '#c0392b'
C_FROZEN = '#7f8c8d'

def _box(ax, x, y, w, h, color, label, sublabel=None, fontsize=8):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                                boxstyle='round,pad=0.05',
                                facecolor=color, edgecolor='white',
                                linewidth=1.5, alpha=0.92))
    ax.text(x, y + (0.18 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize,
            color='white', fontweight='bold')
    if sublabel:
        ax.text(x, y - 0.30, sublabel,
                ha='center', va='center', fontsize=6.5, color='#ecf0f1')

def _arrow(ax, x1, x2, y, color='#7f8c8d'):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# ── CNN ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16); ax.set_ylim(0, 5); ax.axis('off')
Y, W = 2.5, 1.3
cnn_layers = [
    (1.0,  'Input',         '(6, 128)',      C_INPUT),
    (3.0,  'Conv1\n1×9 s2', '(32, 6, 64)',  C_CONV),
    (4.7,  'Pool1\n1×2',    '(32, 6, 32)',  C_POOL),
    (6.4,  'Conv2\n1×3',    '(64, 6, 32)',  C_CONV),
    (8.1,  'Conv3\n1×3',    '(128, 6, 32)', C_CONV),
    (9.8,  'Pool2\n1×2',    '(128, 6, 16)', C_POOL),
    (11.5, 'Conv4\n6×1',    '(128, 1, 16)', C_CONV),
    (13.1, 'Flatten',       '(2048,)',       C_FC),
    (14.8, 'FC',            'n_classes',     C_OUT),
]
for x, label, sublabel, color in cnn_layers:
    _box(ax, x, Y, W, 1.4, color, label, sublabel)
xs = [l[0] for l in cnn_layers]
for i in range(len(xs)-1):
    _arrow(ax, xs[i]+W/2, xs[i+1]-W/2, Y)
ax.annotate('', xy=(11.5, 1.1), xytext=(11.5, Y-0.7),
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5, linestyle='dashed'))
ax.text(11.5, 0.75, 'feature maps (16×128)\n→ LSTM authenticator',
        ha='center', va='top', fontsize=7, color='#e67e22', style='italic')
for xi in [3.0, 6.4, 8.1, 11.5]:
    ax.text(xi, Y+0.9, 'ReLU', ha='center', fontsize=6, color='#bdc3c7')
ax.legend(handles=[
    mpatches.Patch(color=C_CONV, label='Conv2d + ReLU'),
    mpatches.Patch(color=C_POOL, label='MaxPool2d'),
    mpatches.Patch(color=C_FC,   label='Flatten'),
    mpatches.Patch(color=C_OUT,  label='FC (identification)'),
], loc='upper left', fontsize=7, framealpha=0.85, edgecolor='#bdc3c7')
plt.tight_layout(pad=0.3)
plt.savefig(FIG_DIR / 'cnn_architecture.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: cnn_architecture.png')

# ── AuthModel ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6.5))
ax.set_xlim(0, 16); ax.set_ylim(0, 6.5); ax.axis('off')
Y1, Y2, Ym, W = 5.0, 2.0, 3.5, 1.3
for (x, y, label, sublabel, color) in [
    (1.2, Y1, 'x₁\nwindow',   '(6, 128)',  C_INPUT),
    (3.2, Y1, 'CNN\n(frozen)', '(16, 128)', C_FROZEN),
    (1.2, Y2, 'x₂\nwindow',   '(6, 128)',  C_INPUT),
    (3.2, Y2, 'CNN\n(frozen)', '(16, 128)', C_FROZEN),
]:
    _box(ax, x, y, W, 1.3, color, label, sublabel)
_arrow(ax, 1.2+W/2, 3.2-W/2, Y1)
_arrow(ax, 1.2+W/2, 3.2-W/2, Y2)
ax.text(3.2, (Y1+Y2)/2, 'shared\nweights',
        ha='center', va='center', fontsize=6.5, color='#7f8c8d', style='italic')
X_concat = 5.6
for y_src in [Y1, Y2]:
    ax.annotate('', xy=(X_concat-W/2, Ym), xytext=(3.2+W/2, y_src),
                arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=1.5))
merged = [
    (X_concat, 'Concat\nchannel', '(32, 128)', C_FC),
    (7.4,      'LSTM\n2 layers',  '(32, 64)',  C_LSTM),
    (9.2,      'Last\nhidden',    '(64,)',      C_LSTM),
    (11.0,     'Dropout\n+ FC',   '(2,)',       C_FC),
    (12.8,     'Softmax\n[:,1]',  'P(diff)',    C_OUT),
]
for x, label, sublabel, color in merged:
    _box(ax, x, Ym, W, 1.3, color, label, sublabel)
xm = [l[0] for l in merged]
for i in range(len(xm)-1):
    _arrow(ax, xm[i]+W/2, xm[i+1]-W/2, Ym)
ax.text(7.4, Ym-1.0, 'input=128, hidden=64\nnum_layers=2, batch_first=True',
        ha='center', va='top', fontsize=6.5, color='#e67e22', style='italic')
ax.text(12.8, Ym+1.0, 'P(different person) ∈ [0,1]',
        ha='center', va='bottom', fontsize=7, color=C_OUT, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color=C_FROZEN, label='CNN encoder (frozen)'),
    mpatches.Patch(color=C_LSTM,   label='LSTM'),
    mpatches.Patch(color=C_FC,     label='Concat / FC'),
    mpatches.Patch(color=C_OUT,    label='Output score'),
], loc='upper left', fontsize=7, framealpha=0.85, edgecolor='#bdc3c7')
plt.tight_layout(pad=0.3)
plt.savefig(FIG_DIR / 'auth_architecture.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: auth_architecture.png')

/var/folders/vg/cbjg45pn3l10wt3pmdm5gjyw0000gn/T/ipykernel_3338/2792285079.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: cnn_architecture.png
Saved: auth_architecture.png


/var/folders/vg/cbjg45pn3l10wt3pmdm5gjyw0000gn/T/ipykernel_3338/2792285079.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
EPOCH_PATTERN = (
    r'Epoch\s+(?P<epoch>\d+)/\d+'
    r'.*?train_acc=(?P<train_acc>[\d.]+)%'
    r'.*?test_acc=(?P<test_acc>[\d.]+)%'
    r'.*?train_loss=(?P<train_loss>[\d.]+)'
    r'.*?test_loss=(?P<test_loss>[\d.]+)'
)

summary_rows = []
curves = {}

for ds in DATASETS:
    metrics = load_ds_metrics(ds, EXECUTED_DIR)
    if metrics is None:
        print(f'  {ds}: executed notebooks not found — skipping')
        continue

    nb03_path = EXECUTED_DIR / ds / '03_train_authenticator.ipynb'
    epoch_data = parse_epoch_curves(nb03_path, EPOCH_PATTERN)
    if epoch_data:
        curves[ds] = epoch_data

    best_ep = metrics['auth_best_epoch']
    auth_train_acc = np.nan
    if epoch_data and best_ep > 0:
        ep_row = next((d for d in epoch_data if int(d['epoch']) == best_ep), None)
        if ep_row:
            auth_train_acc = ep_row['train_acc']

    overfitting_acc = (auth_train_acc - metrics['auth_test_acc']
                       if not np.isnan(auth_train_acc) and not np.isnan(metrics['auth_test_acc'])
                       else np.nan)

    summary_rows.append({
        'dataset':           ds,
        'cnn_subjects':      metrics['cnn_subjects'],
        'cnn_acc':           metrics['cnn_acc'],
        'cnn_best_epoch':    metrics['cnn_best_epoch'],
        'auth_train_acc':    auth_train_acc,
        'auth_test_acc':     metrics['auth_test_acc'],
        'auth_best_epoch':   best_ep,
        'auth_overfit_acc':  f'+{overfitting_acc:.2f}pp' if not np.isnan(overfitting_acc) else '—',
        'auth_train_auc':    np.nan,
        'auth_test_auc':     metrics['auth_test_auc'],
        'auth_overfit_auc':  '—',
        'auth_same_mean':    metrics['auth_same_mean'],
        'auth_diff_mean':    metrics['auth_diff_mean'],
        'auth_sep':          metrics['auth_sep'],
    })

df = pd.DataFrame(summary_rows).set_index('dataset')
print(df[['cnn_acc', 'auth_train_acc', 'auth_test_acc', 'auth_overfit_acc',
          'auth_test_auc', 'auth_sep']].to_string())
print(f'\nTraining curves loaded: {list(curves.keys())}')

          cnn_acc  auth_train_acc  auth_test_acc auth_overfit_acc  auth_test_auc  auth_sep
dataset                                                                                   
whuGAIT     91.44           91.54          80.12         +11.42pp         0.8698     0.522
ucihar      85.34           81.61          68.83         +12.78pp         0.7388     0.290
wisdm       96.29           98.02          91.87          +6.15pp         0.9577     0.792
combined      NaN           87.36          71.08         +16.28pp         0.7993     0.317

Training curves loaded: ['whuGAIT', 'ucihar', 'wisdm', 'combined']


## Authenticator model

The authenticator is a **frozen CNN + trainable LSTM** stack. The CNN encoder (whuGAIT pre-trained) is kept fixed — it maps each raw 6-channel window (B, 6, 128) to a feature map (B, 16, 128). The LSTM then processes the feature map of **both windows in a pair** and outputs `P(different person)`.

```
Input pair: (window₁, window₂)  →  CNN [frozen] → feature maps
                                 →  2-layer LSTM (hidden=64) → FC(64→2) → softmax
Output: P(same person),  P(different person)
```

Training uses only the LSTM + FC weights (83K parameters), with the CNN's 330K parameters frozen. A pair is classified as **same person** if P(diff) < 0.5.

**Accuracy** = fraction of pairs correctly classified.  
**AUC** = area under the ROC curve treating P(diff) as the score — measures how well the model ranks different-person pairs above same-person pairs independent of the threshold.  
**Score separation** = mean P(diff | diff-pair) − mean P(diff | same-pair) — how far apart the two score distributions are.

In [4]:
avail_ds = [ds for ds in DATASETS if ds in df.index and not np.isnan(df.loc[ds, 'auth_test_acc'])]
x = np.arange(len(avail_ds))
colors = [COLORS[ds] for ds in avail_ds]
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── CNN accuracy ──────────────────────────────────────────────────────────────
ax = axes[0]
cnn_vals = [df.loc[ds, 'cnn_acc'] for ds in avail_ds]
bars = ax.bar(x, cnn_vals, color=colors, alpha=0.85)
ax.axhline(91, ls='--', color='grey', alpha=0.6, label='Paper ref (91%)')
ax.set_xticks(x); ax.set_xticklabels(avail_ds)
ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)')
ax.set_title('CNN encoder — test accuracy')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, cnn_vals):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# ── Auth accuracy: train vs test ──────────────────────────────────────────────
ax = axes[1]
train_vals = [df.loc[ds, 'auth_train_acc'] for ds in avail_ds]
test_vals  = [df.loc[ds, 'auth_test_acc']  for ds in avail_ds]
ax.bar(x - w/2, train_vals, w, color=colors, alpha=0.55, label='Train acc', hatch='//')
ax.bar(x + w/2, test_vals,  w, color=colors, alpha=0.85, label='Test acc')
ax.axhline(93.7, ls='--', color='grey', alpha=0.6, label='Paper ref (93.7%)')
ax.set_xticks(x); ax.set_xticklabels(avail_ds)
ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)')
ax.set_title('Authenticator — train vs test accuracy')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    if not np.isnan(tr):
        ax.text(x[i] - w/2, tr + 1, f'{tr:.1f}%', ha='center', va='bottom', fontsize=8)
    if not np.isnan(te):
        ax.text(x[i] + w/2, te + 1, f'{te:.1f}%', ha='center', va='bottom', fontsize=8)

# ── Auth AUC: train vs test ───────────────────────────────────────────────────
ax = axes[2]
train_auc = [df.loc[ds, 'auth_train_auc'] for ds in avail_ds]
test_auc  = [df.loc[ds, 'auth_test_auc']  for ds in avail_ds]
ax.bar(x - w/2, train_auc, w, color=colors, alpha=0.55, label='Train AUC', hatch='//')
ax.bar(x + w/2, test_auc,  w, color=colors, alpha=0.85, label='Test AUC')
ax.axhline(0.5, ls='--', color='grey', alpha=0.4, label='random')
ax.set_xticks(x); ax.set_xticklabels(avail_ds)
ax.set_ylim(0, 1.1); ax.set_ylabel('AUC')
ax.set_title('Authenticator — train vs test AUC')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
for i, (tr, te) in enumerate(zip(train_auc, test_auc)):
    if not np.isnan(tr):
        ax.text(x[i] - w/2, tr + 0.01, f'{tr:.3f}', ha='center', va='bottom', fontsize=8)
    if not np.isnan(te):
        ax.text(x[i] + w/2, te + 0.01, f'{te:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'models_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/vg/cbjg45pn3l10wt3pmdm5gjyw0000gn/T/ipykernel_3338/2545363956.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Overfitting — per-epoch training curves

Train accuracy rises steadily while test accuracy plateaus early, showing the authenticator fits the training pairs closely but generalises partially. The gap (train − test) is the overfitting measure reported in the LaTeX summary.

In [5]:
if curves:
    n = len(curves)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 8), squeeze=False)

    for col, (ds, data) in enumerate(curves.items()):
        epochs     = [d['epoch']      for d in data]
        train_acc  = [d['train_acc']  for d in data]
        test_acc   = [d['test_acc']   for d in data]
        train_loss = [d['train_loss'] for d in data]
        test_loss  = [d['test_loss']  for d in data]
        best_ep    = int(df.loc[ds, 'auth_best_epoch']) if ds in df.index else None
        c = COLORS[ds]

        # Accuracy
        ax = axes[0][col]
        ax.plot(epochs, train_acc,  color=c,     lw=2,   label='Train acc')
        ax.plot(epochs, test_acc,   color=c,     lw=2,   ls='--', label='Test acc')
        if best_ep:
            ax.axvline(best_ep, ls=':', color='black', alpha=0.5, label=f'Best epoch ({best_ep})')
        ax.set_title(f'{ds}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy (%)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Loss
        ax = axes[1][col]
        ax.plot(epochs, train_loss, color=c,     lw=2,   label='Train loss')
        ax.plot(epochs, test_loss,  color=c,     lw=2,   ls='--', label='Test loss')
        if best_ep:
            ax.axvline(best_ep, ls=':', color='black', alpha=0.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cross-entropy loss')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[0][0].set_ylabel('Accuracy (%)')
    axes[1][0].set_ylabel('Cross-entropy loss')
    plt.suptitle('Authenticator — per-epoch training curves (solid=train, dashed=test)', y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'models_overfitting_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No epoch curves found — run at least one dataset through NB03 first.')

/var/folders/vg/cbjg45pn3l10wt3pmdm5gjyw0000gn/T/ipykernel_3338/3561388327.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Authenticator score separation

How well the model separates same-person from different-person pairs at inference time.

In [6]:
avail = df.dropna(subset=['auth_same_mean', 'auth_diff_mean'])
x     = range(len(avail))

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
ax.bar([i - width/2 for i in x], avail['auth_same_mean'], width,
       label='Same person (low P(diff))', color='#2ecc71', alpha=0.85)
ax.bar([i + width/2 for i in x], avail['auth_diff_mean'], width,
       label='Diff person (high P(diff))', color='#e74c3c', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(avail.index)
ax.set_ylabel('Mean P(different person)')
ax.set_ylim(0, 1)
ax.set_title('Authenticator score separation')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, (_, row) in enumerate(avail.iterrows()):
    ax.annotate(f'sep={row["auth_sep"]:.3f}',
                xy=(i, (row['auth_same_mean'] + row['auth_diff_mean']) / 2),
                ha='center', fontsize=8, color='black')

plt.tight_layout()
plt.savefig(OUT_DIR / 'models_separation.png', dpi=150)
plt.show()

print('\nSummary table:')
print(df[['cnn_acc', 'auth_train_acc', 'auth_test_acc', 'auth_overfit_acc',
          'auth_test_auc', 'auth_sep']].to_string())


Summary table:
          cnn_acc  auth_train_acc  auth_test_acc auth_overfit_acc  auth_test_auc  auth_sep
dataset                                                                                   
whuGAIT     91.44           91.54          80.12         +11.42pp         0.8698     0.522
ucihar      85.34           81.61          68.83         +12.78pp         0.7388     0.290
wisdm       96.29           98.02          91.87          +6.15pp         0.9577     0.792
combined      NaN           87.36          71.08         +16.28pp         0.7993     0.317


/var/folders/vg/cbjg45pn3l10wt3pmdm5gjyw0000gn/T/ipykernel_3338/2270577455.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
from src.utils.latex_writer import write_latex_metrics

def _fmt(v, fmt='.4f'):
    return f'{v:{fmt}}' if isinstance(v, float) and not np.isnan(v) else '—'

for ds in df.index:
    row = df.loc[ds]
    prefix = ds.replace('-', '')
    write_latex_metrics(f'compare_models_{ds}', {
        'cnnTestAcc':     _fmt(row.cnn_acc, '.2f') + ('\\%' if not np.isnan(row.cnn_acc) else ''),
        'authTestAcc':    _fmt(row.auth_test_acc, '.2f') + ('\\%' if not np.isnan(row.auth_test_acc) else ''),
        'authTrainAcc':   _fmt(row.auth_train_acc, '.2f') + ('\\%' if not np.isnan(row.auth_train_acc) else ''),
        'authTestAUC':    _fmt(row.auth_test_auc),
        'authOverfitAcc': str(row.auth_overfit_acc),
        'authSameMean':   _fmt(row.auth_same_mean),
        'authDiffMean':   _fmt(row.auth_diff_mean),
        'authSep':        _fmt(row.auth_sep),
    }, output_dir=str(TEX_DIR), key_prefix=prefix)

print(f'LaTeX written to {TEX_DIR}/ (compare_models_{{ds}}_metrics.tex)')

LaTeX metrics written: ../../latex/generated/compare_models_whuGAIT_metrics.tex
LaTeX metrics written: ../../latex/generated/compare_models_ucihar_metrics.tex
LaTeX metrics written: ../../latex/generated/compare_models_wisdm_metrics.tex
LaTeX metrics written: ../../latex/generated/compare_models_combined_metrics.tex
LaTeX written to ../../latex/generated/ (compare_models_{ds}_metrics.tex)
